# Predict Customer Churn
## Score: .91423

In [10]:
N_FOLDS = 5
N_SEEDS = 3
OPTUNA_TRIALS = 50
USE_RF_ET = False
USE_ORIGINAL_DATA = True
TARGET_ENC_M = 20
BLEND_MODE = 'rank'
USE_ISOTONIC_CALIBRATION = True

In [11]:
import numpy as np
import pandas as pd

train = pd.read_csv("playground-series-s6e3/train.csv")
test = pd.read_csv("playground-series-s6e3/test.csv")
test_ids = test['id']
X_test_raw = test.drop(columns=['id'])

if USE_ORIGINAL_DATA:
    original = pd.read_csv("original_data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
    original = original.drop(columns=['customerID'])
    original = original[train.columns.drop('id')]
    train_combined = pd.concat([train.drop(columns=['id']), original], ignore_index=True)
    sample_weights = np.array([1.0] * len(train) + [0.5] * len(original))
else:
    train_combined = train.drop(columns=['id'])
    sample_weights = np.ones(len(train_combined))

y_full = train_combined['Churn'].map({'Yes': 1, 'No': 0})
X_full = train_combined.drop(columns=['Churn'])
print(f"Train: {len(X_full)}, Original: {USE_ORIGINAL_DATA}")

Train: 601237, Original: True


In [12]:
def engineer_features(df, total_charges_median=None):
    df = df.copy()
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    median_tc = total_charges_median if total_charges_median is not None else df['TotalCharges'].median()
    df['TotalCharges'] = df['TotalCharges'].fillna(median_tc)
    df['AvgChargesPerMonth'] = df['TotalCharges'] / df['tenure'].replace(0, 1)
    df['MonthlyChargesSqrt'] = np.sqrt(df['MonthlyCharges'])
    df['tenure_squared'] = df['tenure'] ** 2
    df['ChargesRatio'] = df['TotalCharges'] / (df['MonthlyCharges'] * (df['tenure'] + 1))
    df['Contract_MonthToMonth'] = (df['Contract'] == 'Month-to-month').astype(int)
    df['Contract_OneYear'] = (df['Contract'] == 'One year').astype(int)
    df['Contract_TwoYear'] = (df['Contract'] == 'Two year').astype(int)
    df['IsFiberOptic'] = (df['InternetService'] == 'Fiber optic').astype(int)
    df['NumStreaming'] = ((df['StreamingTV'] == 'Yes') | (df['StreamingMovies'] == 'Yes')).astype(int)
    df['NumStreamingBoth'] = ((df['StreamingTV'] == 'Yes') & (df['StreamingMovies'] == 'Yes')).astype(int)
    addon_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']
    df['NumAddons'] = sum((df[c] == 'Yes').astype(int) for c in addon_cols)
    df['PaymentElectronic'] = (df['PaymentMethod'] == 'Electronic check').astype(int)
    df['HasDependents'] = (df['Dependents'] == 'Yes').astype(int)
    df['HasPartner'] = (df['Partner'] == 'Yes').astype(int)
    df['SeniorWithShortTenure'] = (df['SeniorCitizen'] == 1) & (df['tenure'] < 12)
    df['SeniorWithShortTenure'] = df['SeniorWithShortTenure'].astype(int)
    df['HighChargeShortTenure'] = (df['MonthlyCharges'] > 70) & (df['tenure'] < 12)
    df['HighChargeShortTenure'] = df['HighChargeShortTenure'].astype(int)
    return df

def target_encode(train_df, test_df, col, target_series, m=5):
    global_mean = target_series.mean()
    combined = pd.DataFrame({col: train_df[col], '_y': target_series.values})
    agg = combined.groupby(col)['_y'].agg(['mean', 'count'])
    smooth = (agg['mean'] * agg['count'] + global_mean * m) / (agg['count'] + m)
    train_df = train_df.copy()
    test_df = test_df.copy()
    train_df[f'{col}_te'] = train_df[col].map(smooth).fillna(global_mean)
    test_df[f'{col}_te'] = test_df[col].map(smooth).fillna(global_mean)
    return train_df, test_df

X_full = engineer_features(X_full)
tc_median = X_full['TotalCharges'].median()
X_test_raw = engineer_features(X_test_raw, total_charges_median=tc_median)

for col in ['Contract', 'PaymentMethod', 'InternetService']:
    X_full, X_test_raw = target_encode(X_full, X_test_raw, col, y_full, m=TARGET_ENC_M)
    X_full = X_full.drop(columns=[col])
    X_test_raw = X_test_raw.drop(columns=[col])

X_encoded = pd.get_dummies(X_full, drop_first=True)
X_test_encoded = pd.get_dummies(X_test_raw, drop_first=True)
X_test_encoded = X_test_encoded.reindex(columns=X_encoded.columns, fill_value=0)

In [13]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fit_kw = {'sample_weight': sample_weights}

In [14]:
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 800),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.15),
        'subsample': trial.suggest_float('subsample', 0.6, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.95),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 4.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
    }
    scores = []
    for train_idx, val_idx in cv.split(X_encoded, y_full):
        X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
        y_tr, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]
        sw_tr = sample_weights[train_idx]
        m = XGBClassifier(**params, random_state=42, eval_metric='logloss')
        m.fit(X_tr, y_tr, sample_weight=sw_tr)
        scores.append(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1]))
    return np.mean(scores)

study_xgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(n_startup_trials=10))
study_xgb.optimize(xgb_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
best_xgb_params = study_xgb.best_params
print(f"XGB Best CV AUC: {study_xgb.best_value:.4f}")

[I 2026-03-09 11:29:48,162] A new study created in memory with name: no-name-0bae3066-8bae-49cd-bf5b-9649e737fe71


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-03-09 11:30:55,994] Trial 0 finished with value: 0.9157148241626414 and parameters: {'n_estimators': 651, 'max_depth': 6, 'learning_rate': 0.06067998287038898, 'subsample': 0.8980977274015395, 'colsample_bytree': 0.7170716421631156, 'scale_pos_weight': 1.012897824510961, 'reg_alpha': 1.8188189790431646, 'reg_lambda': 0.013639133999529995, 'min_child_weight': 1}. Best is trial 0 with value: 0.9157148241626414.
[I 2026-03-09 11:32:37,614] Trial 1 finished with value: 0.9150637305677061 and parameters: {'n_estimators': 732, 'max_depth': 6, 'learning_rate': 0.08865489797661745, 'subsample': 0.7614179457020798, 'colsample_bytree': 0.8034400713388798, 'scale_pos_weight': 3.6242720687385344, 'reg_alpha': 0.009101308626598427, 'reg_lambda': 3.7492493436785486, 'min_child_weight': 6}. Best is trial 0 with value: 0.9157148241626414.
[I 2026-03-09 11:33:10,625] Trial 2 finished with value: 0.9154746411771318 and parameters: {'n_estimators': 342, 'max_depth': 4, 'learning_rate': 0.10695749

In [15]:
from lightgbm import LGBMClassifier

def lgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 800),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.15),
        'subsample': trial.suggest_float('subsample', 0.6, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.95),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 4.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
    }
    scores = []
    for train_idx, val_idx in cv.split(X_encoded, y_full):
        X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
        y_tr, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]
        sw_tr = sample_weights[train_idx]
        m = LGBMClassifier(**params, random_state=42, verbose=-1)
        m.fit(X_tr, y_tr, sample_weight=sw_tr)
        scores.append(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1]))
    return np.mean(scores)

study_lgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(n_startup_trials=10))
study_lgb.optimize(lgb_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
best_lgb_params = study_lgb.best_params
print(f"LGB Best CV AUC: {study_lgb.best_value:.4f}")

[I 2026-03-09 12:19:18,642] A new study created in memory with name: no-name-be072f57-ea6a-4712-8006-f86a10d4457b


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-03-09 12:20:24,626] Trial 0 finished with value: 0.9153438725414744 and parameters: {'n_estimators': 755, 'max_depth': 8, 'learning_rate': 0.03531856487569323, 'subsample': 0.6787072429894985, 'colsample_bytree': 0.8740818718269903, 'scale_pos_weight': 3.4853908255491057, 'reg_alpha': 0.21204650549076573, 'reg_lambda': 3.242381659469335, 'min_child_samples': 19}. Best is trial 0 with value: 0.9153438725414744.
[I 2026-03-09 12:20:57,727] Trial 1 finished with value: 0.9155383150731046 and parameters: {'n_estimators': 411, 'max_depth': 6, 'learning_rate': 0.14256099181949197, 'subsample': 0.7250977359065616, 'colsample_bytree': 0.893020891414275, 'scale_pos_weight': 2.065076444383554, 'reg_alpha': 7.305149117447163, 'reg_lambda': 0.0033818819601519933, 'min_child_samples': 6}. Best is trial 1 with value: 0.9155383150731046.
[I 2026-03-09 12:22:02,181] Trial 2 finished with value: 0.9152895926916711 and parameters: {'n_estimators': 711, 'max_depth': 8, 'learning_rate': 0.03422406

In [16]:
from catboost import CatBoostClassifier

def cat_objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 300, 800),
        'depth': trial.suggest_int('depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.15),
        'subsample': trial.suggest_float('subsample', 0.6, 0.95),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.6, 0.95),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 4.0),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
    }
    scores = []
    for train_idx, val_idx in cv.split(X_encoded, y_full):
        X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
        y_tr, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]
        sw_tr = sample_weights[train_idx]
        m = CatBoostClassifier(**params, random_state=42, verbose=0)
        m.fit(X_tr, y_tr, sample_weight=sw_tr)
        scores.append(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1]))
    return np.mean(scores)

study_cat = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(n_startup_trials=10))
study_cat.optimize(cat_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
best_cat_params = study_cat.best_params
print(f"CatBoost Best CV AUC: {study_cat.best_value:.4f}")

[I 2026-03-09 12:54:50,971] A new study created in memory with name: no-name-e8d1c352-8622-4b50-9655-bfd75bc3d881


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-03-09 12:57:20,911] Trial 0 finished with value: 0.9150069487925041 and parameters: {'iterations': 576, 'depth': 6, 'learning_rate': 0.05226967232602728, 'subsample': 0.9071215056916231, 'colsample_bylevel': 0.8642700548813276, 'scale_pos_weight': 3.749608802031467, 'l2_leaf_reg': 0.0019334008611797025}. Best is trial 0 with value: 0.9150069487925041.
[I 2026-03-09 13:00:22,310] Trial 1 finished with value: 0.9147715040166607 and parameters: {'iterations': 694, 'depth': 7, 'learning_rate': 0.029238222107236304, 'subsample': 0.6183612941951797, 'colsample_bylevel': 0.7058602591507543, 'scale_pos_weight': 3.08939691374526, 'l2_leaf_reg': 0.0013238242276941271}. Best is trial 0 with value: 0.9150069487925041.
[I 2026-03-09 13:02:58,212] Trial 2 finished with value: 0.915583267406902 and parameters: {'iterations': 797, 'depth': 4, 'learning_rate': 0.12694284756365892, 'subsample': 0.8857113508722245, 'colsample_bylevel': 0.8074221924179902, 'scale_pos_weight': 3.4151690034639195, '

In [17]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import roc_auc_score
from sklearn.isotonic import IsotonicRegression
from scipy.optimize import minimize
from scipy.stats import rankdata

n_models = 3 if not USE_RF_ET else 5
oof = np.zeros((len(X_encoded), n_models))
test_preds = np.zeros((len(X_test_encoded), n_models))

def get_models(seed, fold):
    models = [
        XGBClassifier(**best_xgb_params, random_state=seed+fold, eval_metric='logloss'),
        LGBMClassifier(**best_lgb_params, random_state=seed+fold, verbose=-1),
        CatBoostClassifier(**best_cat_params, random_state=seed+fold, verbose=0),
    ]
    if USE_RF_ET:
        models.extend([
            RandomForestClassifier(n_estimators=300, max_depth=12, class_weight='balanced', random_state=seed+fold),
            ExtraTreesClassifier(n_estimators=300, max_depth=12, class_weight='balanced', random_state=seed+fold),
        ])
    return models

for fold, (train_idx, val_idx) in enumerate(cv.split(X_encoded, y_full)):
    X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
    y_tr = y_full.iloc[train_idx]
    sw_tr = sample_weights[train_idx]
    models = get_models(42, fold)
    for i, m in enumerate(models):
        m.fit(X_tr, y_tr, sample_weight=sw_tr)
        oof[val_idx, i] = m.predict_proba(X_val)[:, 1]
        test_preds[:, i] += m.predict_proba(X_test_encoded)[:, 1] / N_FOLDS

def to_rank_percentiles(arr):
    return (rankdata(arr) - 0.5) / len(arr)

if BLEND_MODE == 'rank':
    oof_blend_input = np.column_stack([to_rank_percentiles(oof[:, i]) for i in range(n_models)])
else:
    oof_blend_input = oof.copy()

def neg_auc(w):
    blend = oof_blend_input @ w
    return -roc_auc_score(y_full, blend)

best_auc = -1
best_w = None
for x0 in [np.ones(n_models)/n_models, np.array([0.5, 0.3, 0.2] if n_models==3 else [0.2]*5)]:
    result = minimize(neg_auc, x0=x0, method='SLSQP',
                      bounds=[(0, 1)]*n_models,
                      constraints={'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
    auc = -result.fun
    if auc > best_auc:
        best_auc = auc
        best_w = result.x
blend_weights = best_w
blend_oof = oof_blend_input @ blend_weights

if USE_ISOTONIC_CALIBRATION:
    iso_calib = IsotonicRegression(out_of_bounds='clip')
    iso_calib.fit(blend_oof, y_full)
    blend_oof = np.clip(iso_calib.predict(blend_oof), 0, 1)

names = ['XGB','LGB','Cat'] + (['RF','ET'] if USE_RF_ET else [])
print(f"Blend OOF AUC ({BLEND_MODE}): {roc_auc_score(y_full, blend_oof):.4f}")
print(f"Weights: {dict(zip(names, np.round(blend_weights, 4)))}")

Blend OOF AUC (rank): 0.9161
Weights: {'XGB': np.float64(0.3332), 'LGB': np.float64(0.3334), 'Cat': np.float64(0.3334)}


In [18]:
def blend_test(tp):
    if BLEND_MODE == 'rank':
        tp_input = np.column_stack([to_rank_percentiles(tp[:, i]) for i in range(n_models)])
    else:
        tp_input = tp
    preds = tp_input @ blend_weights
    if USE_ISOTONIC_CALIBRATION:
        preds = np.clip(iso_calib.predict(preds), 0, 1)
    return preds

all_test_preds = [blend_test(test_preds)]
for base_seed in [123, 456, 789, 2024, 2025][:N_SEEDS-1]:
    tp = np.zeros((len(X_test_encoded), n_models))
    for fold, (train_idx, val_idx) in enumerate(cv.split(X_encoded, y_full)):
        X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
        y_tr = y_full.iloc[train_idx]
        sw_tr = sample_weights[train_idx]
        models = get_models(base_seed, fold)
        for i, m in enumerate(models):
            m.fit(X_tr, y_tr, sample_weight=sw_tr)
            tp[:, i] += m.predict_proba(X_test_encoded)[:, 1] / N_FOLDS
    all_test_preds.append(blend_test(tp))

final_preds = np.mean(all_test_preds, axis=0)
submission = pd.DataFrame({'id': test_ids, 'Churn': final_preds})
submission.to_csv('submission.csv', index=False)
submission.head()

,id,Churn
0,594194,0.078015
1,594195,0.000553
2,594196,0.121232
3,594197,0.003419
4,594198,0.510604
